# Lesson 1.1 — Why Integration Is Hard

**Goal:** *experience* the difference between component correctness and system correctness. Two layers that each pass their own tests can still compose into a broken system. We make that happen on purpose.

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, model_perception, understand
checks = []
world = GreenhouseWorld.demo_row(n=6, seed=1)
print('fruit in world:', [f.fid for f in world.fruit])


fruit in world: ['F0', 'F1', 'F2', 'F3', 'F4', 'F5']


### Two layers, each 'correct' in isolation
`perceive` returns detections; `select_nearest` returns the nearest one. Each is trivially correct on its own contract.

In [2]:
def perceive(world):
    return model_perception(world, rng=np.random.default_rng(0))
def select_nearest(dets, tool):
    # CONTRACT: returns nearest detection. Says NOTHING about reachability.
    return min(dets, key=lambda d: np.linalg.norm(d['xy'] - tool))
tool = world.tool_xy()
dets = perceive(world)
nearest = select_nearest(dets, tool)
print('nearest detection:', nearest['id'], 'at', np.round(nearest['xy'],2))
checks.append(nearest['id'] in [d['id'] for d in dets])  # each layer is 'correct'


nearest detection: F3 at [0.45 0.15]


### Composition failure mode 1 — contract violation (reachability)
`select_nearest` can hand the planner an **unreachable** target, because its postcondition never promised reachability. The proper Understand stage (`understand`) closes this seam by also checking reach.

In [3]:
from modules.module09.integration import REACH_MAX
# force an unreachable-but-nearest situation by pushing one fruit just past reach
world.fruit[0].x, world.fruit[0].y = REACH_MAX + 0.3, 0.0
dets = perceive(world)
naive = select_nearest(dets, tool)
_, current = understand(dets, world)   # the seam-aware stage
naive_reach = np.linalg.norm(naive['xy']) <= REACH_MAX
print('naive nearest reachable? ', naive_reach)
print('understand() target    : ', current['id'] if current else None,
      '(reachable)' if current else '')
# the naive composition can select an unreachable target; understand() never does
checks.append(current is None or current['reachable'])


naive nearest reachable?  True
understand() target    :  F3 (reachable)


### Composition failure mode 2 — empty upstream output
If perception returns nothing, `select_nearest` crashes (min of empty). The seam-aware stage degrades gracefully to `None`.

In [4]:
empty = []
crashed = False
try:
    select_nearest(empty, tool)
except ValueError:
    crashed = True
_, current_empty = understand(empty, world)
print('naive on empty crashed:', crashed, '| understand on empty:', current_empty)
checks.append(crashed and current_empty is None)


naive on empty crashed: True | understand on empty: None


In [5]:
# Takeaway: each layer passed its own contract, yet the composition failed in the
# seams (reachability assumption, empty-list handling). Integration owns the seams.
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


3/3 checks passed.
All checks passed.
